# 06 — Create Addresses

For every ACTIVE subscription with a successfully resolved address (from
`05_Fetch_Subscriptions.ipynb`'s Voyager lookup), `PUT`s that address onto
its target account:

```
{{host}}/rest/SubscriberService/v1/subscribers/{accountcode}
```

`addLine1` / `addLine2` / `city` / `zip` / `state` (region ISO) all come
from the Voyager `ParsedAddress_*` columns — real values, not placeholders.

Returns each new address's `id`, saved as `ship_add_id` — this is what
`07_Create_Subscription_Orders.ipynb` uses as `shipAddId` on the order.

Subscriptions whose Voyager lookup didn't resolve are skipped here (flagged
with status `"skipped"`) — they need a manual address before their order can
be created.

Inactive subscriptions don't go through this notebook at all — see
`06_Attach_Inactive_Addresses.ipynb`, which runs AFTER this one finishes
(it needs every active address here to be created first, so it can find
each account's final default service address).


## 1. Setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_addresses")

df_subscriptions = load_subscriptions_resolved()
logger.info(f"Loaded {len(df_subscriptions):,} subscriptions from 05_Fetch_Subscriptions.ipynb")


python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 12
python-dotenv could not parse statement starting at line 17
python-dotenv could not parse statement starting at line 23
python-dotenv could not parse statement starting at line 29
2026-07-28 07:32:16,938 [INFO] Loaded 5 subscriptions from 05_Fetch_Subscriptions.ipynb


## 2. Per-subscription address creation

In [2]:
def create_address_for_subscription(session: requests.Session, row: dict) -> dict:
    subscription_id = row["SubscriptionUSN"]
    account_number  = row["TargetAccountNumber"]

    result = {
        "SubscriptionUSN":      subscription_id,
        "TargetAccountNumber":  account_number,
        "addLine1":             row.get("ParsedAddress_addLine1"),
        "status":               "failed",
        "ship_add_id":          None,
        "error":                None,
    }

    if not row.get("ParsedAddress_parsed_ok"):
        result["status"] = "skipped"
        result["error"] = row.get("ParsedAddress_error") or "Voyager address lookup failed — needs manual address"
        return result

    status, ship_add_id, error = add_address_to_account(
        session,
        account_number,
        row["ParsedAddress_addLine1"],
        location_id=row.get("ParsedAddress_location_id") or str(subscription_id),
        address2=row.get("ParsedAddress_addLine2"),
        city=row.get("ParsedAddress_city") or "Christchurch",
        zip_code=row.get("ParsedAddress_postcode") or "1234",
        region_iso=row.get("ParsedAddress_region_iso"),
    )
    result["status"] = status
    result["ship_add_id"] = ship_add_id
    result["error"] = error

    if status == "created":
        logger.info(f"[OK] subscription {subscription_id} -> {account_number} address id={ship_add_id}")
    else:
        logger.error(f"[FAIL] subscription {subscription_id} — {error}")

    return result


## 3. Run (parallel driver)

In [3]:
def create_all_addresses(df: pd.DataFrame, max_workers: int = MAX_WORKERS) -> pd.DataFrame:
    session = new_session(max_workers=max_workers)
    rows = df.to_dict("records")
    total = len(rows)
    results = []
    logger.info(f"Creating addresses for {total:,} subscriptions with {max_workers} workers...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(create_address_for_subscription, session, row): row["SubscriptionUSN"] for row in rows}
        for i, future in enumerate(as_completed(futures), start=1):
            results.append(future.result())
            if i % 50 == 0 or i == total:
                ok = sum(1 for r in results if r["status"] == "created")
                logger.info(f"Progress: {i}/{total} — {ok} created so far")
    return pd.DataFrame(results)


df_address_results = create_all_addresses(df_subscriptions)
df_address_results.head(20)


2026-07-28 07:32:16,990 [INFO] Creating addresses for 5 subscriptions with 10 workers...
2026-07-28 07:32:24,196 [INFO] [OK] subscription V113063226_fullbatch12 -> 99965692_fullbatch12 address id=142813
2026-07-28 07:32:25,523 [INFO] [OK] subscription V113070452_fullbatch12 -> 99965692_fullbatch12 address id=142814
2026-07-28 07:32:26,692 [INFO] [OK] subscription V113078158_fullbatch12 -> 999656921_fullbatch12 address id=142815
2026-07-28 07:32:27,060 [INFO] [OK] subscription V113064505_fullbatch12 -> 99965692_fullbatch12 address id=142731
2026-07-28 07:32:32,659 [INFO] [OK] subscription V113086607_fullbatch12 -> 999656921_fullbatch12 address id=142816
2026-07-28 07:32:32,663 [INFO] Progress: 5/5 — 5 created so far


,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error
0,V113063226_fullbatch12,99965692_fullbatch12,6/4 JOHNSTON GROVE,created,142813,None
1,V113070452_fullbatch12,99965692_fullbatch12,166A MANCHESTER STREET,created,142814,None
2,V113078158_fullbatch12,999656921_fullbatch12,5/23 ALLEN STREET,created,142815,None
3,V113064505_fullbatch12,99965692_fullbatch12,21/51 TE KANAWA CRESCENT,created,142731,None
4,V113086607_fullbatch12,999656921_fullbatch12,1/245 MOORHOUSE AVENUE,created,142816,None


## 4. Failures / skips

In [4]:
not_created = df_address_results[df_address_results["status"] != "created"]
print(f"{len(not_created):,} / {len(df_address_results):,} addresses not created (failed or skipped)")
not_created.groupby("status").size()


0 / 5 addresses not created (failed or skipped)


Series([], dtype: int64)

## 5. Save

In [5]:
save_df("address_results", df_address_results)


Saved 5 rows -> migration_data\06_address_creation_results.csv
